# A04 — Swaps: vivos, forward-starting, amortizables y basis

**Objetivos:** valorar un swap recién contratado y uno vivo (precio limpio y sucio); swaps forward-starting y
amortizables; basis €STR/Euribor; caso del project finance de 120 M€.

**Fuentes:** Hull cap. 7 y 33; Veronesi cap. 5; Schofield & Bowler cap. 2 y 4. Datos sintéticos.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent          # notebooks/ -> repository root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

from derivlab.market import sample_market
from derivlab.curves import build_curves, forward_rate, discount, roll_curve, shift_curve, tenor_end
from derivlab.dates import spot_date, add_months, add_business_days, year_fraction, make_schedule
from derivlab.swaps import price_swap, par_swap_rate, swap_cashflows, estr_euribor_basis

m = sample_market()
estr, eur = build_curves(m)
ref, spot = m["ref_date"], spot_date(m["ref_date"])

## 1. Swap a 5 años recién contratado

In [2]:
N = 10_000_000
end5 = add_months(spot, 60)
k5 = par_swap_rate(ref, spot, end5, N, estr, eur)
res = price_swap(ref, spot, end5, N, k5, estr, eur, side="payer")
print(f"Par rate {k5:.4%}; PV {res['pv']:.2f}; annuity {res['annuity']:,.0f} EUR per unit of rate")
res["cashflows"][["leg", "accrual_start", "accrual_end", "rate", "tau", "amount", "df", "pv"]].head(8).round(6)

Par rate 2.4040%; PV -0.00; annuity 46,930,460 EUR per unit of rate


,leg,accrual_start,accrual_end,rate,tau,amount,df,pv
0,fixed,2026-10-28,2027-10-28,0.02404,1.000000,-240400.000000,0.981223,-235885.901774
1,fixed,2027-10-28,2028-10-30,0.02404,1.005556,-241735.555556,0.961140,-232341.677085
2,fixed,2028-10-30,2029-10-29,0.02404,0.997222,-239732.222222,0.939678,-225271.067481
3,fixed,2029-10-29,2030-10-28,0.02404,0.997222,-239732.222222,0.917096,-219857.412002
4,fixed,2030-10-28,2031-10-28,0.02404,1.000000,-240400.000000,0.893728,-214852.199757
5,float,2026-10-28,2027-04-28,0.02043,0.505556,103285.000000,0.990532,102307.048534
6,float,2027-04-28,2027-10-28,0.02028,0.508333,103090.000000,0.981223,101154.233003
7,float,2027-10-28,2028-04-28,0.02127,0.508333,108122.500000,0.971467,105037.435111


## 2. Swap vivo: precio limpio y sucio

Payer swap contratado el 28-07-2025 a 5 años sobre 10 M€ al 2,10 %. El Euribor 6M fijó el 28-07-2026 al 2,01 %
(supuesto) para el periodo en curso. Desde el último pago hay intereses devengados en ambas patas.

In [3]:
live_start = date(2025, 7, 30)
live_end = add_months(live_start, 60)
float_periods = make_schedule(live_start, live_end, 6)
current = [p for p in float_periods if p["accrual_start"] < ref < p["accrual_end"]][0]
fixings = {current["fixing_date"]: 0.0201}
live = price_swap(ref, live_start, live_end, N, 0.0210, estr, eur, side="payer", fixings=fixings)
print("Current float period:", current["accrual_start"], "->", current["accrual_end"], "fixed on", current["fixing_date"])
print(f"CASE live swap dirty = {live['pv']:,.0f}  accrued = {live['accrued']:,.0f}  clean = {live['clean']:,.0f}")

Current float period: 2026-07-30 -> 2027-01-29 fixed on 2026-07-28
CASE live swap dirty = 72,743  accrued = -1,033  clean = 73,776


## 3. Forward-starting: 2 años dentro de 5

In [4]:
fs_start = tenor_end(spot, "2Y")
k_fs = par_swap_rate(ref, fs_start, add_months(fs_start, 60), N, estr, eur)
k7 = par_swap_rate(ref, spot, add_months(spot, 84), N, estr, eur)
k2 = par_swap_rate(ref, spot, add_months(spot, 24), N, estr, eur)
print(f"CASE 2y5y forward swap = {k_fs:.4%} vs spot 5y {k5:.4%} (2y {k2:.4%}, 7y {k7:.4%})")

CASE 2y5y forward swap = 2.7060% vs spot 5y 2.4040% (2y 2.1510%, 7y 2.5370%)


## 4. Caso: project finance de 120 M€ a 10 años

El préstamo dispone de 120 M€ hoy, tiene 3 años de carencia y amortiza 15 M€ al final de cada año desde el
tercero. Se compara un swap amortizable que replica el calendario con un swap *bullet* de 120 M€ a 10 años.

In [5]:
total, years = 120e6, 10
amort = {spot: total}
for k in range(3, years):
    amort[add_months(spot, 12 * k)] = total - 15e6 * (k - 2)
end10 = add_months(spot, 120)
notional_path = pd.Series({k: (total if k < 3 else total - 15e6 * (k - 2)) for k in range(years)}, name="notional")
print(notional_path.map("{:,.0f}".format).to_string())

k_amort = par_swap_rate(ref, spot, end10, amort, estr, eur)
k_bullet = par_swap_rate(ref, spot, end10, total, estr, eur)
avg_life = (notional_path.sum()) / total
excess = (total - notional_path).mean()

def pv_amort(e, r, K=k_amort):
    return price_swap(ref, spot, end10, amort, K, e, r)["pv"]
def pv_bullet(e, r, K=k_bullet):
    return price_swap(ref, spot, end10, total, K, e, r)["pv"]
dv01_amort = pv_amort(shift_curve(estr, 1), shift_curve(eur, 1)) - pv_amort(estr, eur)
dv01_bullet = pv_bullet(shift_curve(estr, 1), shift_curve(eur, 1)) - pv_bullet(estr, eur)
print(f"CASE amortising par rate = {k_amort:.4%}; bullet 10y = {k_bullet:.4%}; average life = {avg_life:.2f} years")
print(f"CASE average over-hedge with the bullet swap = {excess / 1e6:.1f} M EUR")
print(f"CASE DV01 amortising = {dv01_amort:,.0f}; bullet = {dv01_bullet:,.0f} EUR/bp")

0    120,000,000
1    120,000,000
2    120,000,000
3    105,000,000
4     90,000,000
5     75,000,000
6     60,000,000
7     45,000,000
8     30,000,000
9     15,000,000
CASE amortising par rate = 2.5351%; bullet 10y = 2.6850%; average life = 6.50 years
CASE average over-hedge with the bullet swap = 42.0 M EUR
CASE DV01 amortising = 72,384; bullet = 107,419 EUR/bp


### Prepago en el año 4

Si la empresa prepaga al final del año 4, el swap amortizable queda vivo sin préstamo debajo y hay que
cancelarlo. Se valora el resto del swap en esa fecha suponiendo que la curva de entonces es la forward de hoy
desplazada −100, 0 o +100 pb.

In [6]:
y4 = add_months(spot, 48)
rest = {d: n for d, n in amort.items() if d >= y4}
rest[y4] = notional_path[4]
rows = []
for shift in [-100, 0, 100]:
    e4 = shift_curve(roll_curve(estr, y4), shift)
    r4 = shift_curve(roll_curve(eur, y4), shift)
    first = make_schedule(y4, end10, 6)[0]
    fix = {first["fixing_date"]: forward_rate(r4, first["accrual_start"], first["accrual_end"])}
    v = price_swap(y4, y4, end10, rest, k_amort, e4, r4, side="payer", fixings=fix)["pv"]
    rows.append({"curve_shift_bp": shift, "unwind_value_eur": v})
unwind = pd.DataFrame(rows)
print(unwind.to_string(index=False, formatters={"unwind_value_eur": "{:,.0f}".format}))
print(f"CASE unwind at year 4 if rates are 100 bp lower = {unwind.unwind_value_eur.iloc[0]:,.0f} EUR")

 curve_shift_bp unwind_value_eur
           -100       -2,002,653
              0        1,048,459
            100        3,973,448
CASE unwind at year 4 if rates are 100 bp lower = -2,002,653 EUR


## 5. Basis €STR / Euribor 6M

In [7]:
b = pd.DataFrame({"tenor_y": [2, 5, 10, 30]})
b["basis_bp"] = [estr_euribor_basis(estr, eur, spot, add_months(spot, 12 * y)) * 1e4 for y in b.tenor_y]
print(b.round(2).to_string(index=False))
print(f"CASE 5y basis = {b.basis_bp.iloc[1]:.1f} bp")

 tenor_y  basis_bp
       2     15.87
       5     13.97
      10     12.51
      30     11.20
CASE 5y basis = 14.0 bp


## 6. Comprobaciones

In [8]:
assert abs(res["pv"]) < 1e-6
assert abs(live["pv"] - live["clean"] - live["accrued"]) < 1e-6
assert k5 < k7 < k_fs                      # upward curve: forward-starting above spot
assert k_amort < k_bullet and abs(dv01_amort) < abs(dv01_bullet)
assert unwind.unwind_value_eur.iloc[0] < 0 < unwind.unwind_value_eur.iloc[2]
print("All checks passed")

All checks passed


## Ejercicios

1. **Básico.** Calcula el tipo par de un swap a 7 años y su anualidad sobre 25 M€.
2. **Intermedio.** Valora un receiver swap a 10 años sobre 50 M€ contratado a mercado al 3,00 % (fuera de mercado). ¿Quién debe dinero a quién?
3. **Avanzado.** El swap vivo de la sección 2 se cancela hoy con la curva desplazada +50 pb. ¿Cuánto cobra o paga la empresa (precio limpio y sucio)?

## Soluciones

In [9]:
r7 = price_swap(ref, spot, add_months(spot, 84), 25e6, 0.0, estr, eur)
print(f"7y par {r7['pv_float'] / r7['annuity']:.4%}, annuity {r7['annuity']:,.0f}")

7y par 2.5370%, annuity 160,210,584


In [10]:
v = price_swap(ref, spot, add_months(spot, 120), 50e6, 0.03, estr, eur, side="receiver")["pv"]
print(f"Receiver at 3.00%: PV {v:,.0f} EUR -> positive: the holder receives above-market fixed, the counterparty owes this value")

Receiver at 3.00%: PV 1,386,387 EUR -> positive: the holder receives above-market fixed, the counterparty owes this value


In [11]:
s = price_swap(ref, live_start, live_end, N, 0.0210, shift_curve(estr, 50), shift_curve(eur, 50),
               side="payer", fixings=fixings)
print(f"+50 bp: dirty {s['pv']:,.0f}, clean {s['clean']:,.0f}")

+50 bp: dirty 240,081, clean 241,115
